In [34]:
import pandas as pd

df = pd.read_csv('001.csv')
df['說明'] = '地址:' + df['地址']+',' + \
'經度:' + df['經度'].astype(str)+',' + \
'緯度:' + df['緯度'].astype(str)+',' + \
'充電樁數量' + df['充電樁數量']

result_df = df[['充電樁位置','說明']]
result_df

,充電樁位置,說明
0,蘆洲三民場,"地址:新北市蘆洲區三民路33號,經度:121.4753234,緯度:25.0832431,充..."
1,南昌場,"地址:臺北市中正區南昌路一段54巷4弄2號,經度:121.5166875,緯度:25.031..."
2,長安站,"地址:臺北市中山區長安東路1段26號,經度:121.5258125,緯度:25.048687..."
3,復興SOGO三站,"地址:臺北市大安區復興南路196巷1號,經度:121.5429375,緯度:25.03993..."
4,中崙站,"地址:臺北市松山區八德路2段174巷11弄5、7號,經度:121.5389375,緯度:25..."
5,青年站,"地址:臺北市萬華區水源路213巷內,經度:121.5013125,緯度:25.0193125..."
6,延吉站,"地址:臺北市松山區八德路3段106巷13號旁,經度:121.5538125,緯度:25.04..."
7,古亭站,"地址:臺北市大安區和平東路1段53號,經度:121.5240625,緯度:25.027187..."
8,士東國小站,"地址:台北市士林區中山北路6段424號,經度:121.5271875,緯度:25.11368..."
9,瑞安站,"地址:臺北市大安區和平東路二段89號,經度:121.5393125,緯度:25.025687..."


In [ ]:
import os
from google import genai
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
for m in client.models.list():
    if m.supported_actions and 'embedContent' in m.supported_actions:
        print(m.name)

In [ ]:
from google.genai import types
model = 'gemini-embedding-001'
def embed_fn(title,text):
    #新版 SDK 不支援 title 參數,將標題併入內容
    result = client.models.embed_content(
        model=model,
        contents=f"{title}\n{text}",
        config=types.EmbedContentConfig(task_type='RETRIEVAL_DOCUMENT')
    )
    return result.embeddings[0].values

result_df['Embeddings'] = result_df.apply(lambda row:embed_fn(row['充電樁位置'],row['說明']),axis=1)
result_df

In [37]:
query = "忠孝玉成站"
model = 'gemini-embedding-001'

In [ ]:
import numpy as np
def find_best_passage(query,dataframe):
    result = client.models.embed_content(
        model=model,
        contents=query,
        config=types.EmbedContentConfig(task_type='RETRIEVAL_QUERY')
    )
    query_embedding = result.embeddings[0].values
    dot_products = np.dot(np.stack(dataframe['Embeddings']),query_embedding)
    print(dot_products)
    idx = np.argmax(dot_products)
    print(idx)

find_best_passage(query,result_df)